# Load FantasyPros ADP (2021-2025)

This notebook pulls FantasyPros NFL ADP snapshots for seasons 2021-2025, combines them into one dataset, and writes local parquet outputs.

In [1]:
%load_ext autotime

time: 0 ns (started: 2026-07-26 15:22:08 -05:00)


In [2]:
from datetime import date
from pathlib import Path

import polars as pl

from nfl.fantasypros_fantasy import FantasyProsApiClient, PipelineConfig, run_pipeline

time: 438 ms (started: 2026-07-26 15:22:08 -05:00)


In [3]:
SEASONS = list(range(2021, 2026))
OUTPUT_DIR = Path("./output/fantasypros_adp_2021_2025")
WRITE_PARQUET = True

# Persist each season through the package pipeline to Iceberg.
RUN_PIPELINE_PERSIST = True
PIPELINE_STORAGE_TARGET = "iceberg"  # one of: none, polars, iceberg, both
PIPELINE_ICEBERG_DRY_RUN = False

print("Seasons:", SEASONS)
print("Output dir:", OUTPUT_DIR)
print("Write parquet:", WRITE_PARQUET)
print("Run pipeline persist:", RUN_PIPELINE_PERSIST)
print("Pipeline storage target:", PIPELINE_STORAGE_TARGET)
print("Pipeline iceberg dry run:", PIPELINE_ICEBERG_DRY_RUN)

Seasons: [2021, 2022, 2023, 2024, 2025]
Output dir: output\fantasypros_adp_2021_2025
Write parquet: True
Run pipeline persist: True
Pipeline storage target: iceberg
Pipeline iceberg dry run: False
time: 0 ns (started: 2026-07-26 15:22:08 -05:00)


In [4]:
client = FantasyProsApiClient(validate_contracts=True)

all_players: list[dict] = []
all_adp: list[dict] = []

for season in SEASONS:
    players = client.get_players(season)
    adp_rows = client.get_adp_snapshots(season, effective_date=date.today())

    all_players.extend(players)
    all_adp.extend(adp_rows)

    print(f"Season {season}: players={len(players)} adp_rows={len(adp_rows)}")

players_df = pl.DataFrame(all_players) if all_players else pl.DataFrame()
adp_df = pl.DataFrame(all_adp) if all_adp else pl.DataFrame()

if players_df.height > 0 and "fp_player_id" in players_df.columns:
    players_df = players_df.unique(subset=["fp_player_id"], keep="first")

if adp_df.height > 0:
    adp_df = adp_df.sort(["season", "rank"])

print("Combined players rows:", players_df.height)
print("Combined ADP rows:", adp_df.height)

ExtractionError: Could not find FantasyPros ADP table with id='data'.

time: 750 ms (started: 2026-07-26 15:22:08 -05:00)


In [ ]:
if adp_df.height == 0:
    print("No ADP rows returned.")
else:
    season_summary = adp_df.group_by("season").agg([
        pl.len().alias("adp_rows"),
        pl.col("fp_player_id").n_unique().alias("unique_players"),
        pl.col("rank").min().alias("min_rank"),
        pl.col("rank").max().alias("max_rank"),
    ]).sort("season")
    print(season_summary)

    adp_with_players_df = adp_df.join(
        players_df.select(["fp_player_id", "full_name", "position", "team"]),
        on="fp_player_id",
        how="left",
    )

    print("Preview:")
    print(adp_with_players_df.select(["season", "rank", "adp", "full_name", "position", "team"]).head(30))

In [ ]:
if WRITE_PARQUET:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    players_path = OUTPUT_DIR / "fantasypros_players_2021_2025.parquet"
    adp_path = OUTPUT_DIR / "fantasypros_adp_2021_2025.parquet"
    joined_path = OUTPUT_DIR / "fantasypros_adp_with_players_2021_2025.parquet"

    players_df.write_parquet(players_path)
    adp_df.write_parquet(adp_path)

    if adp_df.height > 0 and players_df.height > 0:
        adp_with_players_df.write_parquet(joined_path)

    print("Wrote:")
    print("  ", players_path)
    print("  ", adp_path)
    if adp_df.height > 0 and players_df.height > 0:
        print("  ", joined_path)
else:
    print("Parquet write skipped.")

In [ ]:
if RUN_PIPELINE_PERSIST:
    for season in SEASONS:
        result = run_pipeline(
            season=season,
            sport="nfl",
            config=PipelineConfig(
                storage_target=PIPELINE_STORAGE_TARGET,
                effective_date=date.today(),
                iceberg_dry_run=PIPELINE_ICEBERG_DRY_RUN,
                polars_output_dir=OUTPUT_DIR / "pipeline_outputs" / str(season),
            ),
        )
        print(f"Pipeline season {season}: frames={sorted(result.frames.keys())}")
else:
    print("Pipeline persistence skipped. Set RUN_PIPELINE_PERSIST = True to enable.")